# Project 5: Student Performance Predictor
### Multivariate Tabular Modeling & Explainable AI (SHAP)
**Author:** Rafi Kamindra  
**Dataset:** Students Performance Dataset (Kaggle: `rabieelkharoua/students-performance-dataset`)  
**Primary Objectives:** Multivariate Academic Performance Modeling, Target Leakage Prevention, and Explainable AI (XAI) for Data-Driven Educational Interventions

---

## Executive Summary
Predicting student academic performance and identifying at-risk learners early in the academic semester allows educators, counselors, and institutions to take targeted, proactive interventions before academic failure occurs.

Key engineering and research objectives of this final portfolio project:
1. **Target Formulation & Strict Leakage Prevention:** In this dataset, both continuous `GPA` and categorical `GradeClass` are present. To prevent 100% target leakage, `GradeClass` is strictly excluded when predicting `GPA` (and vice-versa).
2. **Model Benchmarking:** Rigorous 5-Fold Cross-Validation comparing Linear/Ridge regression, Decision Tree, Random Forest, and Gradient Boosting (XGBoost).
3. **Explainable AI (XAI):** Applying **SHAP (SHapley Additive exPlanations)** to ensure full transparency. We demonstrate both Global Feature Attribution (identifying key academic drivers) and Local Instance Explanation (explaining individual student risks and formulating actionable interventions).

In [ ]:
# 1. Environment & Library Setup
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment successfully initialized.')

In [ ]:
# 2. Dataset Ingestion & Validation
import glob

# Auto-detect path across Local, Colab, and Kaggle environments
candidate_paths = [
    'student_performance.csv',
    '../student_performance.csv',
    '../../Student_performance_data _.csv',
    'Student_performance_data _.csv',
    '../Student_performance_data _.csv'
]
kaggle_paths = glob.glob('/kaggle/input/**/Student_performance_data _.csv', recursive=True) + glob.glob('/kaggle/input/**/*.csv', recursive=True)

data_path = None
for p in candidate_paths + kaggle_paths:
    if os.path.exists(p) and os.path.isfile(p):
        data_path = p
        break

df = pd.read_csv(data_path)
df = df.dropna().reset_index(drop=True)

print(f'Dataset loaded from: {data_path}')
print(f'Total Students : {df.shape[0]:,}')
print(f'Total Features : {df.shape[1]}')
df.head()

In [ ]:
# 3. Exploratory Data Analysis & Feature Correlation
print('='*60)
print('SUMMARY STATISTICS: GPA (TARGET)')
print('='*60)
print(f'Mean GPA   : {df["GPA"].mean():.2f}')
print(f'Median GPA : {df["GPA"].median():.2f}')
print(f'Std GPA    : {df["GPA"].std():.2f}')
print(f'Min GPA    : {df["GPA"].min():.2f}')
print(f'Max GPA    : {df["GPA"].max():.2f}')
print('='*60)

numeric_cols = df.drop(columns=['StudentID', 'GradeClass']).columns
gpa_corr = df[numeric_cols].corr()['GPA'].sort_values()

print('Feature Correlation with GPA:')
print(gpa_corr.round(3).to_string())

In [ ]:
# 4. Leakage-Free Train-Test Split (80:20)
# Strictly drop StudentID (identifier) and GradeClass (derived target leakage)
X = df.drop(columns=['StudentID', 'GPA', 'GradeClass'])
y = df['GPA']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'Training Set : {X_train.shape[0]:,} students | {X_train.shape[1]} features')
print(f'Holdout Test : {X_test.shape[0]:,} students | {X_test.shape[1]} features')

In [ ]:
# 5. 5-Fold Cross-Validation Model Benchmark
models = {
    '1. Baseline (Ridge Regression)': Ridge(alpha=1.0, random_state=42),
    '2. Decision Tree Regressor': DecisionTreeRegressor(max_depth=5, random_state=42),
    '3. Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    '4. Gradient Boosting (XGBoost)': XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring={
            'r2': 'r2',
            'neg_rmse': 'neg_root_mean_squared_error',
            'neg_mae': 'neg_mean_absolute_error'
        },
        n_jobs=-1
    )
    cv_results.append({
        'Model': name,
        'Mean R2 Score': scores['test_r2'].mean(),
        'Std R2': scores['test_r2'].std(),
        'Mean RMSE': -scores['test_neg_rmse'].mean(),
        'Mean MAE': -scores['test_neg_mae'].mean()
    })

df_cv = pd.DataFrame(cv_results).sort_values(by='Mean R2 Score', ascending=False).reset_index(drop=True)
display(df_cv)

In [ ]:
# 6. Champion Model Training & Holdout Evaluation
champion_model = models['4. Gradient Boosting (XGBoost)']
champion_model.fit(X_train, y_train)

y_pred = champion_model.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)

print(f'Holdout R2 Score : {test_r2:.4f}')
print(f'Holdout RMSE     : {test_rmse:.4f}')
print(f'Holdout MAE      : {test_mae:.4f}')

In [ ]:
# 7. Explainable AI: SHAP Global Attribution & Local Waterfall
explainer = shap.TreeExplainer(champion_model)
shap_values = explainer(X_test)

# Global Beeswarm Summary
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns, show=False)
plt.title('SHAP Global Feature Attribution', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Local Waterfall Plot (Student #0)
plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values[0], max_display=8, show=False)
plt.title('Local SHAP Attribution for Student #0', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Artifact Packaging
output_dir = 'student_performance_artifacts'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(champion_model, os.path.join(output_dir, 'student_model.joblib'))
print('Saved model artifact successfully.')